<a href="https://colab.research.google.com/github/18felasofa/Analisis-Deret-Waktu/blob/main/Tugas_Kelompok_UTS_Data%20Soetta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# TABULASI DATA RUMAH TANGGA + PEUBAH PENYERTA PODES
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. LOKASI FILE
# ============================================================

file_rumah_tangga = r"/content/Peubah_Respon_Rumahtangga.xlsx"
file_podes = r"/content/PeubahPenyerta_Desa.xlsx"

output_file = r"D:\Kuliah\TESIS\SAE\Final\Data\data_gabungan_rumah_tangga_podes.xlsx"


# ============================================================
# 2. MEMBACA DATA
# ============================================================

df_rt = pd.read_excel(file_rumah_tangga)
df_podes = pd.read_excel(file_podes)

print("=" * 70)
print("UKURAN DATA AWAL")
print("=" * 70)

print("Data rumah tangga :", df_rt.shape)
print("Data PODES        :", df_podes.shape)


# ============================================================
# 3. MEMERIKSA VARIABEL UTAMA
# ============================================================

print("\nKolom rumah tangga:")
print(df_rt.columns.tolist())

print("\nKolom PODES:")
print(df_podes.columns.tolist())


# ============================================================
# 4. STANDARDISASI NAMA DESA DAN KECAMATAN
# ============================================================

def standardisasi_nama(x):
    """
    Standardisasi nama wilayah agar:
    - huruf kecil
    - underscore menjadi spasi
    - spasi ganda dihilangkan
    - spasi awal/akhir dihilangkan
    """

    if pd.isna(x):
        return np.nan

    x = str(x).lower().strip()
    x = x.replace("_", " ")
    x = " ".join(x.split())

    return x


df_rt["Desa_std"] = df_rt["Desa"].apply(standardisasi_nama)
df_rt["Kecamatan_std"] = df_rt["Kecamatan"].apply(standardisasi_nama)

df_podes["Desa_std"] = df_podes["Desa"].apply(standardisasi_nama)
df_podes["Kecamatan_std"] = df_podes["NAMA_KEC"].apply(standardisasi_nama)


# ============================================================
# 5. MEMBUAT KUNCI DESA + KECAMATAN
# ============================================================

df_rt["KEY_DESA"] = (
    df_rt["Kecamatan_std"] + "|" + df_rt["Desa_std"]
)

df_podes["KEY_DESA"] = (
    df_podes["Kecamatan_std"] + "|" + df_podes["Desa_std"]
)


print("\nContoh KEY_DESA rumah tangga:")
display(
    df_rt[
        ["Kecamatan", "Desa", "KEY_DESA"]
    ].head(10)
)

print("\nContoh KEY_DESA PODES:")
display(
    df_podes[
        ["NAMA_KEC", "Desa", "KEY_DESA"]
    ].head(10)
)


# ============================================================
# 6. MEMERIKSA DUPLIKASI DATA PODES
# ============================================================

duplikat_key = df_podes["KEY_DESA"].duplicated(keep=False)

print("\n" + "=" * 70)
print("PEMERIKSAAN DUPLIKASI PODES")
print("=" * 70)

print(
    "Jumlah baris PODES dengan KEY_DESA duplikat:",
    duplikat_key.sum()
)

print(
    "Jumlah KEY_DESA unik:",
    df_podes["KEY_DESA"].nunique()
)

if duplikat_key.sum() > 0:

    print("\nPERHATIAN:")
    print(
        "Masih terdapat KEY_DESA yang duplikat. "
        "Periksa data berikut:"
    )

    display(
        df_podes.loc[
            duplikat_key,
            [
                "IDDESA",
                "NAMA_KEC",
                "Desa",
                "KEY_DESA"
            ]
        ].sort_values("KEY_DESA")
    )


# ============================================================
# 7. MEMERIKSA APAKAH IDDESA UNIK
# ============================================================

print("\n" + "=" * 70)
print("PEMERIKSAAN IDDESA")
print("=" * 70)

print(
    "Jumlah IDDESA:",
    df_podes["IDDESA"].nunique()
)

print(
    "Jumlah baris PODES:",
    len(df_podes)
)

if df_podes["IDDESA"].duplicated().sum() > 0:

    print("\nPERINGATAN: IDDESA masih memiliki duplikasi.")

    display(
        df_podes[
            df_podes["IDDESA"].duplicated(keep=False)
        ].sort_values("IDDESA")
    )


# ============================================================
# 8. MEMILIH PEUBAH PENYERTA PODES
# ============================================================

# X1 sampai X15 adalah peubah penyerta
peubah_penyerta = [
    f"X{i}" for i in range(1, 16)
]

print("\nPeubah penyerta yang digunakan:")
print(peubah_penyerta)


# ============================================================
# 9. MEMBUAT DATA PODES YANG AKAN DIGABUNGKAN
# ============================================================

kolom_podes = [
    "KEY_DESA",
    "IDDESA",
    "NAMA_KEC",
    "Desa"
] + peubah_penyerta

df_podes_join = df_podes[kolom_podes].copy()


# ============================================================
# 10. MERGE DATA RUMAH TANGGA DENGAN PODES
# ============================================================

df_gabungan = pd.merge(
    df_rt,
    df_podes_join,
    on="KEY_DESA",
    how="left",
    validate="many_to_one",
    indicator=True
)


# ============================================================
# 11. HASIL MERGE
# ============================================================

print("\n" + "=" * 70)
print("HASIL PENGGABUNGAN")
print("=" * 70)

print("Jumlah rumah tangga awal :", len(df_rt))
print("Jumlah data setelah merge:", len(df_gabungan))

print("\nStatus penggabungan:")
print(
    df_gabungan["_merge"].value_counts()
)


# ============================================================
# 12. RUMAH TANGGA YANG TIDAK MENDAPAT PEUBAH PODES
# ============================================================

tidak_match = df_gabungan[
    df_gabungan["_merge"] == "left_only"
].copy()

print("\nJumlah rumah tangga yang tidak mendapatkan data PODES:")
print(len(tidak_match))


if len(tidak_match) > 0:

    print("\nDaftar desa yang tidak berhasil dicocokkan:")

    display(
        tidak_match[
            [
                "Kecamatan",
                "Desa",
                "KEY_DESA"
            ]
        ].drop_duplicates()
        .sort_values(["Kecamatan", "Desa"])
    )


# ============================================================
# 13. RUMAH TANGGA YANG BERHASIL DIGABUNGKAN
# ============================================================

match = df_gabungan[
    df_gabungan["_merge"] == "both"
].copy()

print("\nJumlah rumah tangga berhasil digabungkan:")
print(len(match))


# ============================================================
# 14. MENAMPILKAN TABULASI RUMAH TANGGA
# ============================================================

kolom_tabulasi = [
    "IDDESA",
    "Kecamatan",
    "Desa",
    "Jumlah_ART",
    "Y_perkapita"
] + peubah_penyerta

kolom_tabulasi = [
    x for x in kolom_tabulasi
    if x in df_gabungan.columns
]

print("\n" + "=" * 70)
print("TABULASI DATA RUMAH TANGGA + PODES")
print("=" * 70)

display(
    df_gabungan[
        kolom_tabulasi
    ].head(20)
)


# ============================================================
# 15. MEMERIKSA CONTEXTUAL VARIABLE
# ============================================================

print("\n" + "=" * 70)
print("PEMERIKSAAN CONTEXTUAL VARIABLES")
print("=" * 70)

for var in peubah_penyerta:

    if var in df_gabungan.columns:

        cek = (
            df_gabungan
            .groupby("KEY_DESA")[var]
            .nunique(dropna=True)
        )

        jumlah_berbeda = (cek > 1).sum()

        print(
            f"{var}: "
            f"{jumlah_berbeda} desa memiliki nilai berbeda "
            f"antar rumah tangga"
        )


# ============================================================
# 16. MEMBUAT TABULASI LEVEL DESA
# ============================================================

tabulasi_desa = (
    df_gabungan
    .groupby(
        [
            "IDDESA",
            "Kecamatan",
            "Desa"
        ],
        dropna=False
    )
    .agg(
        n_rumah_tangga=(
            "Y_perkapita",
            "count"
        ),

        mean_pengeluaran=(
            "Y_perkapita",
            "mean"
        ),

        var_pengeluaran=(
            "Y_perkapita",
            "var"
        ),

        sd_pengeluaran=(
            "Y_perkapita",
            "std"
        )
    )
    .reset_index()
)


# ============================================================
# 17. MENAMBAHKAN PEUBAH PODES LEVEL DESA
# ============================================================

contextual_desa = (
    df_podes_join
    .drop_duplicates("IDDESA")
    [
        [
            "IDDESA"
        ] + peubah_penyerta
    ]
)

tabulasi_desa = pd.merge(
    tabulasi_desa,
    contextual_desa,
    on="IDDESA",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 18. MENAMPILKAN TABULASI LEVEL DESA
# ============================================================

print("\n" + "=" * 70)
print("TABULASI LEVEL DESA")
print("=" * 70)

display(
    tabulasi_desa.head(20)
)


# ============================================================
# 19. MENGHAPUS KOLOM PEMBANTU
# ============================================================

df_gabungan = df_gabungan.drop(
    columns=["_merge"],
    errors="ignore"
)


# ============================================================
# 20. MENYIMPAN HASIL KE EXCEL
# ============================================================

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    df_gabungan.to_excel(
        writer,
        sheet_name="RumahTangga_Contextual",
        index=False
    )

    tabulasi_desa.to_excel(
        writer,
        sheet_name="Tabulasi_Desa",
        index=False
    )

    df_podes_join.to_excel(
        writer,
        sheet_name="PODES_Desa",
        index=False
    )


print("\n" + "=" * 70)
print("SELESAI")
print("=" * 70)

print("File hasil:")
print(output_file)


UKURAN DATA AWAL
Data rumah tangga : (500, 169)
Data PODES        : (280, 21)

Kolom rumah tangga:
['Nama Surveyor', 'Tanggal Survey', 'Nama', 'Jenis Kelamin', 'Alamat', 'R103', 'Kecamatan', 'Desa', 'RT/RW', 'No. Telp/HP', 'Tanggal Lahir', 'Pendidikan terakhir', 'Pekerjaan', 'Lainnya...14', 'Tempat Bekerja', 'Apakah dalam satu tahun terakhir ada anggota keluarga dibawah umur 1 tahun yang meninggal?', 'Karbohidrat', 'Protein', 'SayurandanKacang', 'Buah', 'Rokok', 'Lainnya_Makanan', 'Waktu...23', 'Nilai (Rp)...24', 'Frekuensi dalam setahun (untuk pilihan waktu bulanan)...25', 'Waktu...26', 'Nilai (Rp)...27', 'Frekuensi dalam setahun (untuk pilihan waktu bulanan)...28', 'Waktu...29', 'Nilai (Rp)...30', 'Frekuensi dalam setahun (untuk pilihan waktu bulanan)...31', 'Waktu...32', 'Nilai (Rp)...33', 'Frekuensi dalam setahun (untuk pilihan waktu bulanan)...34', 'Waktu...35', 'Nilai (Rp)...36', 'Frekuensi dalam setahun (untuk pilihan waktu bulanan)...37', 'Waktu...38', 'Nilai (Rp)...39', 'Freku

,Kecamatan,Desa,KEY_DESA
0,Cileunyi,Cileunyi Kulon,cileunyi|cileunyi kulon
1,Kutawaringin,Jatisari_Kutawaringin,kutawaringin|jatisari kutawaringin
2,Cileunyi,Cileunyi Kulon,cileunyi|cileunyi kulon
3,Cileunyi,Cileunyi_Kulon,cileunyi|cileunyi kulon
4,Margahayu,Sukamenak,margahayu|sukamenak
5,Kutawaringin,Jatisari_Kutawaringin,kutawaringin|jatisari kutawaringin
6,Cileunyi,Cileunyi_Kulon,cileunyi|cileunyi kulon
7,Cileunyi,Cileunyi Kulon,cileunyi|cileunyi kulon
8,Kutawaringin,Jatisari_Kutawaringin,kutawaringin|jatisari kutawaringin
9,Margahayu,Sukamenak,margahayu|sukamenak



Contoh KEY_DESA PODES:


,NAMA_KEC,Desa,KEY_DESA
0,Ciwidey,Panundaan,ciwidey|panundaan
1,Ciwidey,Ciwidey,ciwidey|ciwidey
2,Ciwidey,Panyocokan,ciwidey|panyocokan
3,Ciwidey,Lebakmuncang,ciwidey|lebakmuncang
4,Ciwidey,Rawabogo,ciwidey|rawabogo
5,Ciwidey,Nengkelan,ciwidey|nengkelan
6,Ciwidey,Sukawening,ciwidey|sukawening
7,Rancabali,Cipelah,rancabali|cipelah
8,Rancabali,Sukaresmi,rancabali|sukaresmi
9,Rancabali,Indragiri,rancabali|indragiri



PEMERIKSAAN DUPLIKASI PODES
Jumlah baris PODES dengan KEY_DESA duplikat: 0
Jumlah KEY_DESA unik: 280

PEMERIKSAAN IDDESA
Jumlah IDDESA: 280
Jumlah baris PODES: 280

Peubah penyerta yang digunakan:
['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15']

HASIL PENGGABUNGAN
Jumlah rumah tangga awal : 500
Jumlah data setelah merge: 500

Status penggabungan:
_merge
both          470
left_only      30
right_only      0
Name: count, dtype: int64

Jumlah rumah tangga yang tidak mendapatkan data PODES:
30

Daftar desa yang tidak berhasil dicocokkan:


KeyError: "['Desa'] not in index"

In [ ]:
# ============================================================
# 1. IMPORT PACKAGE
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 2. LOKASI FILE
# ============================================================

file_rumah_tangga = r"/content/Peubah_Respon_Rumahtangga.xlsx"
file_podes = r"/content/PeubahPenyerta_Desa.xlsx"

output_file = r"D:\Kuliah\TESIS\SAE\Final\Data\data_gabungan_rumah_tangga_podes.xlsx"


# ============================================================
# 3. MEMBACA DATA
# ============================================================

df_rt = pd.read_excel(file_rumah_tangga)
df_podes = pd.read_excel(file_podes)

print("Ukuran data rumah tangga :", df_rt.shape)
print("Ukuran data PODES       :", df_podes.shape)

print("\nKolom data rumah tangga:")
print(df_rt.columns.tolist())

print("\nKolom data PODES:")
print(df_podes.columns.tolist())

Ukuran data rumah tangga : (500, 169)
Ukuran data PODES       : (280, 21)

Kolom data rumah tangga:
['Nama Surveyor', 'Tanggal Survey', 'Nama', 'Jenis Kelamin', 'Alamat', 'R103', 'Kecamatan', 'Desa', 'RT/RW', 'No. Telp/HP', 'Tanggal Lahir', 'Pendidikan terakhir', 'Pekerjaan', 'Lainnya...14', 'Tempat Bekerja', 'Apakah dalam satu tahun terakhir ada anggota keluarga dibawah umur 1 tahun yang meninggal?', 'Karbohidrat', 'Protein', 'SayurandanKacang', 'Buah', 'Rokok', 'Lainnya_Makanan', 'Waktu...23', 'Nilai (Rp)...24', 'Frekuensi dalam setahun (untuk pilihan waktu bulanan)...25', 'Waktu...26', 'Nilai (Rp)...27', 'Frekuensi dalam setahun (untuk pilihan waktu bulanan)...28', 'Waktu...29', 'Nilai (Rp)...30', 'Frekuensi dalam setahun (untuk pilihan waktu bulanan)...31', 'Waktu...32', 'Nilai (Rp)...33', 'Frekuensi dalam setahun (untuk pilihan waktu bulanan)...34', 'Waktu...35', 'Nilai (Rp)...36', 'Frekuensi dalam setahun (untuk pilihan waktu bulanan)...37', 'Waktu...38', 'Nilai (Rp)...39', 'Frek

In [ ]:
# Melihat beberapa baris pertama
display(df_rt.head())
display(df_podes.head())

,Nama Surveyor,Tanggal Survey,Nama,Jenis Kelamin,Alamat,R103,Kecamatan,Desa,RT/RW,No. Telp/HP,...,_notes,_status,_submitted_by,__version__,_tags,_index,Jumlah_ART,Total_Makanan_Mingguan,Total_Makanan_Bulanan,Y_perkapita
0,Fajryanti Kusuma Wardani,2023-07-10,Ayi Karmini,Perempuan,Jl Ciburial Kampung Negasari,290,Cileunyi,Cileunyi Kulon,01/20,NaN,...,NaN,submitted_via_web,NaN,vGKRqTe5yzPTRssR7MCCZV,NaN,1,6,278500,1.206833e+06,201138.888889
1,Khairani Cahyoja Utami,2023-07-10,Anrohana,Perempuan,Gang Babakan Cipedung No. 58,191,Kutawaringin,Jatisari_Kutawaringin,01/01,NaN,...,NaN,submitted_via_web,NaN,vGKRqTe5yzPTRssR7MCCZV,NaN,2,3,480000,2.080000e+06,693333.333333
2,Fajryanti Kusuma Wardani,2023-07-10,Elis,Perempuan,Jl Neglasari 3,290,Cileunyi,Cileunyi Kulon,01/20,NaN,...,NaN,submitted_via_web,NaN,vGKRqTe5yzPTRssR7MCCZV,NaN,3,4,333000,1.443000e+06,360750.000000
3,Fajryanti Kusuma Wardani,2023-07-10,Ghozali,NaN,Kp Neglasari,290,Cileunyi,Cileunyi_Kulon,1/20,NaN,...,NaN,submitted_via_web,NaN,v9Dpbr96qGh64DHgkRgxYi,NaN,4,5,1015000,4.398333e+06,879666.666667
4,Febrina Nurhijah,2023-07-09,Cucu Nurhayati,Perempuan,Jalan Sumintapura,260,Margahayu,Sukamenak,05/10,089688618697,...,NaN,submitted_via_web,NaN,vGKRqTe5yzPTRssR7MCCZV,NaN,5,3,210000,9.100000e+05,303333.333333


,IDDESA,R103,R104,NAMA_KEC,Desa,total_keluarga,X1,X2,X3,X4,...,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15
0,3204010001,10,1,Ciwidey,Panundaan,4453,100.0,0.000000,0.0,0.011228,...,0.000449,0.011453,0.002470,0.000000,0.001123,0.000000,0.000674,0.034808,0.000000,0.000000
1,3204010002,10,2,Ciwidey,Ciwidey,4825,100.0,0.414508,0.0,0.016580,...,0.001658,0.064663,0.001451,0.000207,0.003523,0.000622,0.000415,0.041244,0.000622,0.000622
2,3204010003,10,3,Ciwidey,Panyocokan,4891,100.0,0.000000,0.0,0.013085,...,0.000204,0.023104,0.000000,0.000000,0.002249,0.000000,0.000000,0.043345,0.000000,0.000204
3,3204010004,10,4,Ciwidey,Lebakmuncang,5882,100.0,0.000000,0.0,0.010031,...,0.000170,0.010371,0.001360,0.000000,0.001190,0.000000,0.000000,0.005100,0.000000,0.000170
4,3204010005,10,5,Ciwidey,Rawabogo,2280,100.0,0.000000,0.0,0.010965,...,0.000000,0.037281,0.000000,0.000000,0.003070,0.000000,0.000000,0.003947,0.000000,0.000439


In [ ]:
# Cek tipe data Desa
print("Tipe Desa rumah tangga :", df_rt["Desa"].dtype)
print("Tipe Desa PODES        :", df_podes["Desa"].dtype)

Tipe Desa rumah tangga : object
Tipe Desa PODES        : object


In [ ]:
df_rt["Desa"] = (
    df_rt["Desa"]
    .astype(str)
    .str.strip()
)

df_podes["Desa"] = (
    df_podes["Desa"]
    .astype(str)
    .str.strip()
)

In [ ]:
duplikat_podes = (
    df_podes["Desa"]
    .duplicated(keep=False)
)

print("Jumlah baris PODES yang memiliki Desa duplikat:",
      duplikat_podes.sum())

display(
    df_podes.loc[duplikat_podes]
    .sort_values("Desa")
)

print("\nMenampilkan semua kolom untuk baris dengan Desa duplikat:")
with pd.option_context('display.max_columns', None):
    display(
        df_podes.loc[duplikat_podes]
        .sort_values("Desa")
    )

Jumlah baris PODES yang memiliki Desa duplikat: 50


,IDDESA,R103,R104,NAMA_KEC,Desa,total_keluarga,X1,X2,X3,X4,...,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15
143,3204120011,120,11,Majalaya,Bojong,6421,100.000000,0.000000,0.000000,0.036131,...,0.000311,0.018377,0.000000,0.000000,0.000779,0.000000,0.000000,0.022894,0.000000,0.000156
112,3204101002,101,2,Nagreg,Bojong,2147,100.000000,0.000000,0.000000,0.050769,...,0.000000,0.027946,0.000000,0.000000,0.001863,0.000000,0.000000,0.018631,0.000000,0.000000
130,3204110012,110,12,Rancaekek,Cangkuang,4712,100.000000,0.000000,0.000000,0.000637,...,0.000000,0.009975,0.000000,0.000000,0.000637,0.000000,0.000000,0.067912,0.000212,0.000000
200,3204161006,161,6,Cangkuang,Cangkuang,3158,100.000000,7.504750,0.000000,0.045915,...,0.000633,0.066498,0.000317,0.000000,0.000950,0.000317,0.000317,0.079481,0.000317,0.000317
17,3204020006,20,6,Pasirjambu,Cibodas,2929,100.000000,0.000000,0.000000,0.006145,...,0.000341,0.005804,0.000000,0.000000,0.004097,0.000000,0.000000,0.020143,0.000000,0.000341
231,3204191007,191,7,Kutawaringin,Cibodas,2628,100.000000,0.000000,0.000000,0.008371,...,0.000000,0.009513,0.000000,0.000000,0.001903,0.000381,0.000000,0.019026,0.000000,0.000000
146,3204121003,121,3,Solokan jeruk,Cibodas,3781,100.000000,3.464692,0.000000,0.004761,...,0.000000,0.030944,0.000000,0.000000,0.003438,0.000000,0.000000,0.075112,0.000000,0.002116
152,3204130002,130,2,Ciparay,Cikoneng,2025,100.000000,0.000000,0.000000,0.025679,...,0.001975,0.010864,0.000494,0.000000,0.004938,0.000000,0.000000,0.172840,0.000000,0.000000
21,3204020010,20,10,Pasirjambu,Cikoneng,2108,100.000000,0.000000,0.000000,0.120493,...,0.000000,0.008065,0.000000,0.000000,0.005218,0.000000,0.000000,0.067837,0.000000,0.000000
232,3204191008,191,8,Kutawaringin,Jatisari,4024,100.000000,0.000000,0.000000,0.023111,...,0.000000,0.004970,0.000000,0.000000,0.001988,0.000000,0.000249,0.013419,0.000000,0.000000



Menampilkan semua kolom untuk baris dengan Desa duplikat:


,IDDESA,R103,R104,NAMA_KEC,Desa,total_keluarga,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15
143,3204120011,120,11,Majalaya,Bojong,6421,100.000000,0.000000,0.000000,0.036131,0.018689,0.000311,0.018377,0.000000,0.000000,0.000779,0.000000,0.000000,0.022894,0.000000,0.000156
112,3204101002,101,2,Nagreg,Bojong,2147,100.000000,0.000000,0.000000,0.050769,0.042385,0.000000,0.027946,0.000000,0.000000,0.001863,0.000000,0.000000,0.018631,0.000000,0.000000
130,3204110012,110,12,Rancaekek,Cangkuang,4712,100.000000,0.000000,0.000000,0.000637,0.043506,0.000000,0.009975,0.000000,0.000000,0.000637,0.000000,0.000000,0.067912,0.000212,0.000000
200,3204161006,161,6,Cangkuang,Cangkuang,3158,100.000000,7.504750,0.000000,0.045915,0.079164,0.000633,0.066498,0.000317,0.000000,0.000950,0.000317,0.000317,0.079481,0.000317,0.000317
17,3204020006,20,6,Pasirjambu,Cibodas,2929,100.000000,0.000000,0.000000,0.006145,0.032776,0.000341,0.005804,0.000000,0.000000,0.004097,0.000000,0.000000,0.020143,0.000000,0.000341
231,3204191007,191,7,Kutawaringin,Cibodas,2628,100.000000,0.000000,0.000000,0.008371,0.015221,0.000000,0.009513,0.000000,0.000000,0.001903,0.000381,0.000000,0.019026,0.000000,0.000000
146,3204121003,121,3,Solokan jeruk,Cibodas,3781,100.000000,3.464692,0.000000,0.004761,0.009257,0.000000,0.030944,0.000000,0.000000,0.003438,0.000000,0.000000,0.075112,0.000000,0.002116
152,3204130002,130,2,Ciparay,Cikoneng,2025,100.000000,0.000000,0.000000,0.025679,0.064198,0.001975,0.010864,0.000494,0.000000,0.004938,0.000000,0.000000,0.172840,0.000000,0.000000
21,3204020010,20,10,Pasirjambu,Cikoneng,2108,100.000000,0.000000,0.000000,0.120493,0.025142,0.000000,0.008065,0.000000,0.000000,0.005218,0.000000,0.000000,0.067837,0.000000,0.000000
232,3204191008,191,8,Kutawaringin,Jatisari,4024,100.000000,0.000000,0.000000,0.023111,0.037773,0.000000,0.004970,0.000000,0.000000,0.001988,0.000000,0.000249,0.013419,0.000000,0.000000


In [ ]:
df_podes['Desa_Kecamatan'] = df_podes['Desa'] + ' - ' + df_podes['NAMA_KEC']

print("Kolom 'Desa_Kecamatan' berhasil ditambahkan.")
display(df_podes[['Desa', 'NAMA_KEC', 'Desa_Kecamatan']].head())

Kolom 'Desa_Kecamatan' berhasil ditambahkan.


,Desa,NAMA_KEC,Desa_Kecamatan
0,Panundaan,Ciwidey,Panundaan - Ciwidey
1,Ciwidey,Ciwidey,Ciwidey - Ciwidey
2,Panyocokan,Ciwidey,Panyocokan - Ciwidey
3,Lebakmuncang,Ciwidey,Lebakmuncang - Ciwidey
4,Rawabogo,Ciwidey,Rawabogo - Ciwidey


In [ ]:
df_rt['Desa'] = df_rt['Desa'].astype(str).str.lower().str.replace('_', ' ').str.strip()
df_rt['Kecamatan'] = df_rt['Kecamatan'].astype(str).str.lower().str.replace('_', ' ').str.strip()
df_rt['Desa_Kecamatan'] = df_rt['Desa'] + ' - ' + df_rt['Kecamatan']

print("Kolom 'Desa_Kecamatan' berhasil ditambahkan ke df_rt setelah standardisasi.")
display(df_rt[['Desa', 'Kecamatan', 'Desa_Kecamatan']].head())

Kolom 'Desa_Kecamatan' berhasil ditambahkan ke df_rt setelah standardisasi.


,Desa,Kecamatan,Desa_Kecamatan
0,cileunyi kulon,cileunyi,cileunyi kulon - cileunyi
1,jatisari kutawaringin,kutawaringin,jatisari kutawaringin - kutawaringin
2,cileunyi kulon,cileunyi,cileunyi kulon - cileunyi
3,cileunyi kulon,cileunyi,cileunyi kulon - cileunyi
4,sukamenak,margahayu,sukamenak - margahayu


In [ ]:
df_gabungan = pd.merge(df_rt, df_podes, on='Desa_Kecamatan', how='left')

print("DataFrame berhasil digabungkan. Berikut adalah beberapa baris pertama dari DataFrame gabungan:")
display(df_gabungan.head())
print("Ukuran DataFrame gabungan:", df_gabungan.shape)

DataFrame berhasil digabungkan. Berikut adalah beberapa baris pertama dari DataFrame gabungan:


,Nama Surveyor,Tanggal Survey,Nama,Jenis Kelamin,Alamat,R103_x,Kecamatan,Desa_x,RT/RW,No. Telp/HP,...,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15
0,Fajryanti Kusuma Wardani,2023-07-10,Ayi Karmini,Perempuan,Jl Ciburial Kampung Negasari,290,cileunyi,cileunyi kulon,01/20,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Khairani Cahyoja Utami,2023-07-10,Anrohana,Perempuan,Gang Babakan Cipedung No. 58,191,kutawaringin,jatisari kutawaringin,01/01,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fajryanti Kusuma Wardani,2023-07-10,Elis,Perempuan,Jl Neglasari 3,290,cileunyi,cileunyi kulon,01/20,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fajryanti Kusuma Wardani,2023-07-10,Ghozali,NaN,Kp Neglasari,290,cileunyi,cileunyi kulon,1/20,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Febrina Nurhijah,2023-07-09,Cucu Nurhayati,Perempuan,Jalan Sumintapura,260,margahayu,sukamenak,05/10,089688618697,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Ukuran DataFrame gabungan: (500, 191)


In [ ]:
nan_in_iddesa = df_gabungan['Desa'].isna().sum()

print(f"Jumlah baris di df_gabungan yang memiliki NaN pada kolom 'IDDESA' (tidak ditemukan di df_podes): {nan_in_iddesa}")

KeyError: 'Desa'

In [ ]:
pip install thefuzz fuzzywuzzy[speedup]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.8 MB/s eta 0:00:00


In [ ]:
from thefuzz import fuzz
from thefuzz import process

# Membuat list unik dari Desa_Kecamatan dari df_podes untuk dicocokkan
podes_desa_kecamatan_unique = df_podes['Desa_Kecamatan'].unique().tolist()

# Fungsi untuk menemukan kecocokan fuzzy terbaik
def find_best_match(target_string, choices, threshold=80):
    if pd.isna(target_string):
        return None, 0
    # Menggunakan process.extractOne untuk efisiensi
    # target_string: string yang ingin dicocokkan
    # choices: daftar string yang akan dicocokkan
    # scorer: fungsi untuk menghitung skor kesamaan
    # score_cutoff: skor minimum agar dianggap cocok
    best_match = process.extractOne(target_string, choices, scorer=fuzz.token_sort_ratio, score_cutoff=threshold)
    if best_match:
        return best_match[0], best_match[1]
    return None, 0

# Menerapkan fungsi pencocokan fuzzy ke df_rt
# Ini mungkin memakan waktu, jadi ada baiknya mencetak progress
print("Mulai proses fuzzy matching, ini mungkin memakan waktu...")
df_rt[['Fuzzy_Matched_Desa_Kecamatan', 'Fuzzy_Score']] = df_rt['Desa_Kecamatan'].apply(lambda x: pd.Series(find_best_match(x, podes_desa_kecamatan_unique)))

print("Fuzzy matching selesai.")
display(df_rt[['Desa_Kecamatan', 'Fuzzy_Matched_Desa_Kecamatan', 'Fuzzy_Score']].head())

Mulai proses fuzzy matching, ini mungkin memakan waktu...
Fuzzy matching selesai.


,Desa_Kecamatan,Fuzzy_Matched_Desa_Kecamatan,Fuzzy_Score
0,cileunyi kulon - cileunyi,Cileunyi kulon - Cileunyi,100.0
1,jatisari kutawaringin - kutawaringin,Kutawaringin - Kutawaringin,85.0
2,cileunyi kulon - cileunyi,Cileunyi kulon - Cileunyi,100.0
3,cileunyi kulon - cileunyi,Cileunyi kulon - Cileunyi,100.0
4,sukamenak - margahayu,Sukamenak - Margahayu,100.0


In [ ]:
# Melakukan penggabungan ulang dengan hasil fuzzy matching
df_gabungan_fuzzy = pd.merge(
    df_rt,
    df_podes,
    left_on='Fuzzy_Matched_Desa_Kecamatan',
    right_on='Desa_Kecamatan',
    how='left',
    suffixes=('_rt', '_podes')
)

print("DataFrame berhasil digabungkan dengan fuzzy matching.")
display(df_gabungan_fuzzy.head())
print("Ukuran DataFrame gabungan fuzzy:", df_gabungan_fuzzy.shape)

DataFrame berhasil digabungkan dengan fuzzy matching.


,Nama Surveyor,Tanggal Survey,Nama,Jenis Kelamin,Alamat,R103_rt,Kecamatan,Desa_rt,RT/RW,No. Telp/HP,...,X7,X8,X9,X10,X11,X12,X13,X14,X15,Desa_Kecamatan_podes
0,Fajryanti Kusuma Wardani,2023-07-10,Ayi Karmini,Perempuan,Jl Ciburial Kampung Negasari,290,cileunyi,cileunyi kulon,01/20,NaN,...,0.004890,0.000136,0.0,0.002988,0.000000,0.000272,0.069682,0.000272,0.000407,Cileunyi kulon - Cileunyi
1,Khairani Cahyoja Utami,2023-07-10,Anrohana,Perempuan,Gang Babakan Cipedung No. 58,191,kutawaringin,jatisari kutawaringin,01/01,NaN,...,0.005407,0.000000,0.0,0.001802,0.000000,0.000000,0.011896,0.000360,0.000000,Kutawaringin - Kutawaringin
2,Fajryanti Kusuma Wardani,2023-07-10,Elis,Perempuan,Jl Neglasari 3,290,cileunyi,cileunyi kulon,01/20,NaN,...,0.004890,0.000136,0.0,0.002988,0.000000,0.000272,0.069682,0.000272,0.000407,Cileunyi kulon - Cileunyi
3,Fajryanti Kusuma Wardani,2023-07-10,Ghozali,NaN,Kp Neglasari,290,cileunyi,cileunyi kulon,1/20,NaN,...,0.004890,0.000136,0.0,0.002988,0.000000,0.000272,0.069682,0.000272,0.000407,Cileunyi kulon - Cileunyi
4,Febrina Nurhijah,2023-07-09,Cucu Nurhayati,Perempuan,Jalan Sumintapura,260,margahayu,sukamenak,05/10,089688618697,...,0.003094,0.000000,0.0,0.001494,0.000107,0.000640,0.011737,0.000213,0.000320,Sukamenak - Margahayu


Ukuran DataFrame gabungan fuzzy: (500, 194)


In [ ]:
# Memeriksa kembali jumlah NaN pada kolom IDDESA setelah fuzzy matching
nan_in_iddesa_fuzzy = df_gabungan_fuzzy['IDDESA'].isna().sum()

print(f"Jumlah baris di df_gabungan_fuzzy yang memiliki NaN pada kolom 'IDDESA' (tidak ditemukan di df_podes): {nan_in_iddesa_fuzzy}")

Jumlah baris di df_gabungan_fuzzy yang memiliki NaN pada kolom 'IDDESA' (tidak ditemukan di df_podes): 5


Jika jumlah NaN berkurang, berarti fuzzy matching berhasil menemukan lebih banyak kecocokan. Kita juga bisa memeriksa kolom `Fuzzy_Score` untuk memahami kualitas kecocokan.